# Implementando Multi-Query Retriever en tu códigoAquí tienes el fragmento de código adaptado a tu entorno. Puedes probarlo en un script suelto o integrarlo en tu proyecto actual:

In [ ]:
import loggingfrom langchain_ollama import ChatOllamafrom langchain_huggingface import HuggingFaceEmbeddingsfrom langchain_chroma import Chromafrom langchain.retrievers.multi_query import MultiQueryRetriever# 1. Configurar logging (Opcional pero muy recomendado)# Esto te permitirá ver en la terminal las 3 preguntas alternativas que Llama 3 se inventalogging.basicConfig()logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)# 2. Inicializar tu LLM (Llama 3 en Ollama)llm = ChatOllama(model="llama3", temperature=0) # Temperatura 0 para que sea preciso# 3. Inicializar tu modelo de Embeddingsembeddings = HuggingFaceEmbeddings(    model_name="paraphrase-multilingual-MiniLM-L12-v2")# 4. Conectar a tu Chroma DB existente# Asumiendo que tu base de datos está guardada en la carpeta "./chroma_db"vector_db = Chroma(    persist_directory="./chroma_db",     embedding_function=embeddings)# El retriever clásico (RAG 1.0) que trae los 3 mejores resultadosbase_retriever = vector_db.as_retriever(search_kwargs={"k": 3})# 5. LA MAGIA RAG 2.0: Envolverlo en el Multi-Queryretriever_avanzado = MultiQueryRetriever.from_llm(    retriever=base_retriever,    llm=llm)# 6. Prueba en acciónpregunta_usuario = "¿Cómo gestiono las variables de entorno en este proyecto?"print("Buscando en la base de datos con Multi-Query...")# Al usar invoke, Llama 3 generará variaciones, Chroma buscará, y LangChain unirá los resultados sin duplicadosdocumentos_recuperados = retriever_avanzado.invoke(pregunta_usuario)print(f"\n¡Éxito! Se han recuperado {len(documentos_recuperados)} documentos únicos para el contexto.")for i, doc in enumerate(documentos_recuperados):    print(f"\n--- Documento {i+1} ---")    print(doc.page_content[:200] + "...") # Imprime los primeros 200 caracteres

## ¿Qué está pasando bajo el capó?1. Cuando ejecutas invoke(), LangChain le envía un prompt oculto a tu Llama 3 en Ollama diciendo algo como: "Eres un asistente de IA. Tu tarea es generar 3 versiones diferentes de la pregunta del usuario para buscar en una base de datos vectorial...".2. Llama 3 devuelve 3 preguntas diferentes (ej. ¿Dónde está el archivo .env?, ¿Configuración de variables de entorno?, etc.).3. Tu modelo MiniLM convierte esas 3 preguntas nuevas + la original en embeddings.3. ChromaDB hace 4 búsquedas independientes.5. LangChain coge todos los resultados, elimina los fragmentos de texto duplicados y te devuelve una lista limpia de documentos, lista para pasársela de nuevo a Llama 3 para que genere la respuesta final.

## Añadiendo la cadena de generación finalVamos a cerrar el círculo. Esta es la pieza final donde la magia del sistema RAG realmente cobra vida, conectando las búsquedas con el cerebro analítico de tu Llama 3 local.Para lograr esto, vamos a utilizar LCEL (LangChain expresión Language). Es la forma moderna y eficiente en la que LangChain une diferentes componentes usando el símbolo | (como si fueran tuberías por las que fluye la información).Añade los siguientes módulos de LangChain en la parte superior de tu archivo y luego pega este código justo debajo del script que hicimos en el paso anterior:

In [ ]:
from langchain_core.prompts import PromptTemplatefrom langchain_core.output_parsers import StrOutputParserfrom langchain_core.runnables import RunnablePassthrough# 1. Crear el Prompt (Las instrucciones estrictas para Llama 3)# Es crucial decirle que solo use el contexto para evitar "alucinaciones"template = """Usa los siguientes fragmentos de contexto recuperados para responder a la pregunta. Si no sabes la respuesta basándote EXCLUSIVAMENTE en este contexto, di que no lo sabes. No inventes información. Responde de forma clara y directa.Contexto recuperado:{context}Pregunta del usuario: {question}Respuesta:"""prompt = PromptTemplate.from_template(template)# 2. Función auxiliar para formatear los documentos# LangChain recupera objetos, pero Llama 3 solo entiende texto plano.# Esta función extrae el contenido de cada documento y los une con saltos de línea.def format_docs(docs):    return "\n\n".join(doc.page_content for doc in docs)# 3. Construir la cadena principal (La "tubería" RAG)rag_chain = (    # A. Prepara las dos variables que necesita el prompt: context y question    {"context": retriever_avanzado | format_docs, "question": RunnablePassthrough()}    # B. Pasa esas variables al PromptTemplate    | prompt    # C. Se lo envía a Llama 3 (tu modelo en Ollama)    | llm    # D. Extrae solo el texto de la respuesta final, ignorando metadatos extra    | StrOutputParser())# 4. ¡Ejecutar el sistema completo!pregunta_usuario = "¿Cómo gestiono las variables de entorno en este proyecto?"print("Analizando la pregunta, buscando en ChromaDB y generando la respuesta final...\n")# Usamos invoke() sobre la cadena completa, no solo sobre el retrieverrespuesta_final = rag_chain.invoke(pregunta_usuario)print("--- Respuesta de Llama 3 ---")print(respuesta_final)

## ¿Cómo funciona esta "tubería" exactamente?La clave de este código está en el paso 3 (rag_chain). Cuando tú llamas a rag_chain.invoke(pregunta_usuario), ocurre lo siguiente en cadena:1. RunnablePassthrough() toma tu pregunta original y la deja pasar tal cual hacia el prompt.2. Al mismo tiempo, tu pregunta viaja hacia retriever_avanzado, donde Llama 3 genera las 3 variaciones, busca en ChromaDB y devuelve los documentos.3. Esos documentos pasan por format_docs, convirtiéndose en un bloque de texto gigante.4. Tanto la pregunta como el bloque de texto se inyectan en tu prompt.5. Todo ese paquete de texto se envía a llm (Llama 3 en Ollama) para que razone y redacte.6. Finalmente, StrOutputParser() limpia la salida y te la imprime en la pantalla lista para leer.Con esto ya tienes un sistema funcional mucho más robusto que el clásico. Evitarás respuestas vacías porque el Multi-Query Retriever se ha asegurado de peinar bien tu base de datos antes de responder.

## Implementar un Re-ranker (Fase 2) es probablemente el cambio que mayor impacto visual tiene en la calidad de las respuestas de un RAG.En la Fase 1 hicimos que el sistema buscara "a lo ancho" (usando múltiples preguntas). Ahora vamos a poner un "juez" estricto que evalúe todo lo que se ha encontrado y filtre la basura antes de que Llama 3 lo lea.Dado que estás trabajando con un entorno 100% local, usaremos un Cross-Encoder de la librería de HuggingFace que ya tienes instalada. Un modelo muy recomendable y ligero para esto es BAAI/bge-reranker-base.Implementando el Re-ranker en tu tubería (LCEL)Para que esto funcione con el código que ya tienes, vamos a usar lo que LangChain llama un ContextualCompressionRetriever. Básicamente, "comprime" una lista larga de documentos dejando solo los más relevantes.Añade estas importaciones al principio de tu archivo:

In [ ]:
from langchain.retrievers import ContextualCompressionRetrieverfrom langchain.retrievers.document_compressors import CrossEncoderRerankerfrom langchain_community.cross_encoders import HuggingFaceCrossEncoder

Y aquí tienes cómo integrarlo en el flujo que construimos antes. Fíjate en cómo encadenamos las cosas:

In [ ]:
# 1. Configurar el modelo "Juez" (Cross-Encoder local)# La primera vez que ejecutes esto, descargará el modelo (aprox. 1GB)modelo_juez = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")# 2. Configurar el compresor# Le decimos al juez que, de todos los documentos que reciba, solo pase los 3 mejores a Llama 3compresor = CrossEncoderReranker(model=modelo_juez, top_n=3)# 3. Ajustar el Retriever Base (¡Importante!)# Como ahora tenemos un juez filtrando, necesitamos darle MÁS opciones iniciales.# Cambiamos la búsqueda en ChromaDB para que traiga 10 documentos en lugar de 3.base_retriever = vector_db.as_retriever(search_kwargs={"k": 10})# (Aquí iría tu MultiQueryRetriever de la Fase 1, usando este nuevo base_retriever)# retriever_avanzado = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)# 4. Crear el Retriever final con Re-ranking# Envolvemos el retriever_avanzado (Fase 1) con el compresor (Fase 2)retriever_filtrado = ContextualCompressionRetriever(    base_compressor=compresor,    base_retriever=retriever_avanzado )# 5. Actualizar la cadena RAG# Simplemente cambiamos el retriever en nuestra tubería LCELrag_chain = (    # AHORA USAMOS EL RETRIEVER FILTRADO    {"context": retriever_filtrado | format_docs, "question": RunnablePassthrough()}    | prompt    | llm    | StrOutputParser())# Ejecutamos igual que antesprint("Buscando, filtrando con el juez, y generando respuesta...\n")respuesta_final = rag_chain.invoke("¿Cómo gestiono las variables de entorno en este proyecto?")print(respuesta_final)

¿Por qué esto es tan poderoso?Sin el Re-ranker, ChromaDB busca por "similitud de vectores" (qué tan cerca están las palabras conceptualmente). Pero a veces, un texto que tiene palabras muy similares no responde a la pregunta real.El Cross-Encoder lee la pregunta del usuario y el fragmento de texto al mismo tiempo, y genera una puntuación de relevancia lógica. Al decirle a ChromaDB que busque 10 documentos (o incluso 20) por cada variación de la pregunta, y luego pasar todos esos documentos por el Cross-Encoder para que elija los 3 mejores absolutos, le estás entregando a Llama 3 un contexto increíblemente puro y preciso.    ⚠️ Ten en cuenta: Añadir el Re-ranker suma un paso más de procesamiento por red neuronal. Si notas que tu portátil se ralentiza demasiado, puedes cambiar BAAI/bge-reranker-base por un modelo aún más pequeño como cross-encoder/ms-marco-MiniLM-L-6-v2.

Ver las puntuaciones en tiempo real es fundamental cuando estás construyendo un RAG, porque te permite saber si el modelo "juez" está haciendo bien su trabajo o si necesitas afinar tu base de datos.El "truco" es sorprendentemente sencillo. Cuando el CrossEncoderReranker procesa los documentos y los filtra, inyecta silenciosamente la puntuación exacta dentro de los metadatos de cada documento antes de pasárselos al LLM.Para ver esto en acción, en lugar de ejecutar toda la cadena (rag_chain), vamos a invocar solo al retriever filtrado y a escanear lo que nos devuelve.Puedes añadir este pequeño bloque de código al final de tu script para probarlo:

In [ ]:
pregunta = "¿Cómo gestiono las variables de entorno en este proyecto?"print("Buscando, generando variaciones y pasando por el Juez...\n")# 1. Invocamos SOLO al retriever filtrado (sin pasarlo a Llama 3 todavía)documentos_finales = retriever_filtrado.invoke(pregunta)print(f"El Juez ha seleccionado los {len(documentos_finales)} mejores documentos:\n")# 2. Bucle para imprimir el contenido y la puntuación secretafor i, doc in enumerate(documentos_finales):    # La puntuación se guarda en el diccionario de metadatos bajo la clave 'relevance_score'    score = doc.metadata.get('relevance_score', 0.0)        # Imprimimos la posición, la puntuación formateada y un extracto del texto    print(f"--- Top {i+1} | Puntuación de Relevancia: {score:.4f} ---")    print(f"Texto: {doc.page_content[:150]}...\n")

¿Cómo interpretar estas puntuaciones?Las puntuaciones que verás en la terminal dependen del modelo exacto de Cross-Encoder que estés usando (en nuestro caso, BAAI/bge-reranker-base), pero generalmente funcionan así:    Puntuaciones altas (ej. > 0.5 o > 0.8): El juez está muy seguro de que ese fragmento de texto responde directamente a la pregunta.    Puntuaciones bajas o negativas (ej. < 0.1 o números en negativo): El texto comparte palabras con la pregunta, pero el juez ha detectado que el significado o el contexto no tienen nada que ver.Si haces varias pruebas, verás que ChromaDB a veces extrae un documento porque tiene la palabra "variables", pero el juez le pone una nota bajísima y lo descarta porque habla de "variables de CSS" en lugar de "variables de entorno (.env)". ¡Ahí es donde ves brillar a la Fase 2!

¡Llegamos a la joya de la corona! La Fase 3 es donde tu sistema deja de ser un simple buscador glorificado y se convierte en un pequeño "agente" capaz de tomar decisiones.El problema del RAG tradicional es que es rígido: si el usuario dice "Hola, buenos días", el sistema intentará buscar "Hola, buenos días" en tu base de datos de código, no encontrará nada útil, y el LLM probablemente responda algo confuso o te diga que no lo sabe.Para solucionar esto, vamos a implementar un Enrutador Lógico (Router) usando LCEL. Le enseñaremos a Llama 3 a analizar primero la intención del usuario y decidir qué "tubería" usar:    La ruta de la base de datos (nuestro RAG avanzado).    La ruta de conversación general (tirar de la memoria base del LLM).Implementando el Router Agéntico en tu códigoAñade estas importaciones a tu script. Usaremos RunnableLambda para crear la lógica de decisión:

In [ ]:
from langchain_core.runnables import RunnableLambda# 1. Crear el "Cerebro Clasificador"# Le pedimos a Llama 3 que actúe como un semáforo antes de hacer nada más.prompt_clasificador = PromptTemplate.from_template(    """Analiza la siguiente pregunta del usuario y clasifícala en UNA de estas dos categorías:    - 'repositorio': Si la pregunta es sobre código, archivos, variables de entorno, configuraciones o el proyecto.    - 'general': Si es un saludo, una despedida, o una pregunta de cultura general.        Responde ÚNICAMENTE con la palabra de la categoría exacta. No des explicaciones.        Pregunta: {question}    Categoría:""")# Esta pequeña cadena solo devuelve la palabra "repositorio" o "general"cadena_clasificadora = prompt_clasificador | llm | StrOutputParser()# 2. Preparar la "Ruta Alternativa" (Conversación general)# (Asumimos que tu 'rag_chain' de la Fase 2 ya está creada y lista para ser la Ruta Principal)prompt_general = PromptTemplate.from_template(    "Eres un asistente de IA amigable. Responde a esto de forma natural: {question}")cadena_general = prompt_general | llm | StrOutputParser()# 3. La función de enrutamiento (El Agente en acción)def enrutar_pregunta(info):    categoria = info["categoria"].strip().lower()    pregunta = info["question"]        if "repositorio" in categoria:        print("\n🤖 [AGENTE] He detectado una pregunta técnica. Consultando ChromaDB...")        # Llamamos a la tubería compleja que hicimos en la Fase 2        return rag_chain    else:        print("\n🤖 [AGENTE] He detectado charla general. Respondiendo de memoria...")        # Llamamos a la tubería sencilla        return cadena_general# 4. Construir el Flujo Agéntico Finalflujo_agentico = (    # Primero, pasamos la pregunta por el clasificador para obtener la categoría,     # y mantenemos la pregunta original viva para el siguiente paso.    {        "categoria": cadena_clasificadora,        "question": lambda x: x["question"]     }    # Luego, pasamos ese diccionario a nuestra función de enrutamiento    | RunnableLambda(enrutar_pregunta))# --- ¡Hora de probar la inteligencia de tu sistema! ---preguntas_prueba = [    "¡Hola! ¿Qué tal estás hoy?",    "¿Cómo gestiono las variables de entorno en este proyecto?"]for pregunta in preguntas_prueba:    print(f"\n--- Nueva consulta: '{pregunta}' ---")        # Invocamos el flujo maestro en lugar de la cadena RAG directamente    # Pasamos un diccionario porque nuestra cadena ahora espera una clave "question" desde el principio    respuesta = flujo_agentico.invoke({"question": pregunta})        print("\nRespuesta Final:")    print(respuesta)    print("-" * 40)

¿Por qué esto cambia las reglas del juego?    Ahorro de recursos y tiempo: Si el usuario hace una pregunta general, te saltas por completo el Multi-Query, el Embedding y el Re-ranking (el modelo Juez). La respuesta es casi instantánea.    Modularidad absoluta: Imagina que mañana añades una base de datos en SQL o buscas en la web. Solo tendrías que decirle al clasificador: "Elige entre: repositorio, general, o base_datos_sql", y añadir un simple elif en tu función enrutar_pregunta. Es la base de los agentes complejos.    Reducción de alucinaciones: Al aislar el RAG solo para preguntas relevantes, evitas que Llama 3 intente forzar el contexto de tu código para responder a un "Buenos días".Con todo esto (Multi-Query + Re-ranking + Router Agéntico), has pasado de un script básico a una arquitectura RAG 2.0 avanzada digna de producción, todo corriendo en local en tu portátil.

Aquí tienes el código completo, unificado, limpio y comentado. He simplificado un poco la parte final del "Agente" usando una función estándar de Python en lugar de forzarlo todo en LangChain (LCEL), porque en entornos reales es mucho más fácil de leer, mantener y depurar así.Copia este código en un archivo nuevo (por ejemplo, rag_agente.py) en tu portátil:

In [ ]:
import loggingfrom langchain_ollama import ChatOllamafrom langchain_huggingface import HuggingFaceEmbeddingsfrom langchain_chroma import Chromafrom langchain.retrievers.multi_query import MultiQueryRetrieverfrom langchain.retrievers import ContextualCompressionRetrieverfrom langchain.retrievers.document_compressors import CrossEncoderRerankerfrom langchain_community.cross_encoders import HuggingFaceCrossEncoderfrom langchain_core.prompts import PromptTemplatefrom langchain_core.output_parsers import StrOutputParserfrom langchain_core.runnables import RunnablePassthrough# Opcional: Silenciar los logs de LangChain para mantener la terminal limpialogging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)# ==========================================# CONFIGURACIÓN INICIAL (Modelos y Base de Datos)# ==========================================print("Cargando modelos... (Esto puede tardar unos segundos la primera vez)")# 1. Tu LLLocalllm = ChatOllama(model="llama3", temperature=0)# 2. Modelo de Embeddingsembeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")# 3. Base de datos vectorial (Asegúrate de que la ruta "./chroma_db" sea la correcta)vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)# ==========================================# FASE 1 y 2: RAG AVANZADO (Búsqueda + Juez)# ==========================================# A. Retriever Base: Buscamos 10 documentos para tener margen de filtradobase_retriever = vector_db.as_retriever(search_kwargs={"k": 10})# B. Fase 1 (Multi-Query): Llama 3 inventa variaciones de la preguntaretriever_expandido = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)# C. Fase 2 (Re-ranking): El "Juez" evalúa y se queda con los 3 mejoresmodelo_juez = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")compresor = CrossEncoderReranker(model=modelo_juez, top_n=3)# Unimos Fase 1 y Fase 2retriever_filtrado = ContextualCompressionRetriever(    base_compressor=compresor,    base_retriever=retriever_expandido)# D. Cadena RAG Final (La que lee los documentos y responde)def format_docs(docs):    return "\n\n".join(doc.page_content for doc in docs)prompt_rag = PromptTemplate.from_template(    """Usa ÚNICAMENTE el siguiente contexto para responder a la pregunta.     Si no lo sabes, di que no tienes información suficiente. No inventes.        Contexto: {context}        Pregunta: {question}    Respuesta:""")# Tubería RAG (Buscar -> Formatear -> Prompt -> Llama 3 -> Texto)cadena_rag = (    {"context": retriever_filtrado | format_docs, "question": RunnablePassthrough()}    | prompt_rag    | llm    | StrOutputParser())# ==========================================# FASE 3: EL AGENTE (Enrutador)# ==========================================# A. El Clasificador (El "Cerebro" que decide qué ruta tomar)prompt_clasificador = PromptTemplate.from_template(    """Clasifica la siguiente entrada del usuario en UNA de estas dos categorías:    - 'repositorio': Si trata sobre código, variables, errores, archivos o el proyecto.    - 'general': Si es un saludo, charla trivial o pregunta genérica.        Responde SÓLO con la palabra de la categoría (repositorio o general).        Entrada: {question}    Categoría:""")cadena_clasificadora = prompt_clasificador | llm | StrOutputParser()# B. Ruta alternativa (Conversación general sin buscar en base de datos)prompt_general = PromptTemplate.from_template("Eres un asistente amigable. Responde: {question}")cadena_general = prompt_general | llm | StrOutputParser()# C. Función Principal del Agentedef agente_inteligente(pregunta: str):    print(f"\n[Usuario]: {pregunta}")    print("🤖 Analizando intención...")        # 1. Clasificamos la pregunta    categoria = cadena_clasificadora.invoke({"question": pregunta}).strip().lower()        # 2. Tomamos la decisión    if "repositorio" in categoria:        print("🔍 [Ruta] Pregunta técnica detectada. Buscando en la base de datos y filtrando...")        respuesta = cadena_rag.invoke(pregunta)    else:        print("💬 [Ruta] Conversación general. Respondiendo de memoria...")        respuesta = cadena_general.invoke({"question": pregunta})            print(f"\n[Asistente]: {respuesta}\n")    print("-" * 50)# ==========================================# EJECUCIÓN DEL SCRIPT# ==========================================if __name__ == "__main__":    print("\n🚀 Sistema RAG 2.0 Agéntico Iniciado 🚀")    print("-" * 50)        # Pruebas de fuego    agente_inteligente("¡Hola! ¿Me puedes ayudar con unas dudas?")    agente_inteligente("¿Cómo gestiono las variables de entorno en este proyecto?")

Pasos previos antes de ejecutarlo:    Asegúrate de que Ollama está ejecutándose en tu portátil y que tienes el modelo descargado (ollama run llama3).    Confirma que la carpeta ./chroma_db (o el nombre que le hayas puesto a tu base de datos) está en la misma ubicación desde donde vas a ejecutar el script.    Ejecútalo en la terminal de VS Code con: python rag_agente.py.El script descargará el modelo del "Juez" (bge-reranker-base) la primera vez que lo lances, así que tardará un poquito más, pero luego irá mucho más fluido.

Añadir memoria es el toque final para que deje de ser un simple buscador y se convierta en un compañero de programación o de trabajo con el que puedes mantener una conversación natural.Para que Llama 3 recuerde lo que habéis hablado, no necesitamos una base de datos compleja. Nos basta con guardar el historial en una lista en Python e inyectar esos mensajes anteriores dentro de las instrucciones (prompts) justo antes de hacerle la nueva pregunta. De esta manera, si tú le dices "¿Y cómo ejecuto ese archivo?", el LLM mirará el historial, sabrá de qué archivo estáis hablando, buscará la respuesta y te contestará con coherencia.Aquí tienes la versión final del script con el historial de chat (memoria) integrado. He destacado los cambios en la sección de los prompts y en la función del agente:

In [ ]:
import loggingfrom langchain_ollama import ChatOllamafrom langchain_huggingface import HuggingFaceEmbeddingsfrom langchain_chroma import Chromafrom langchain.retrievers.multi_query import MultiQueryRetrieverfrom langchain.retrievers import ContextualCompressionRetrieverfrom langchain.retrievers.document_compressors import CrossEncoderRerankerfrom langchain_community.cross_encoders import HuggingFaceCrossEncoderfrom langchain_core.prompts import PromptTemplatefrom langchain_core.output_parsers import StrOutputParserfrom langchain_core.runnables import RunnablePassthroughlogging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)print("Cargando modelos... (Esto puede tardar unos segundos)")llm = ChatOllama(model="llama3", temperature=0)embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)# ==========================================# FASES 1 y 2: RAG AVANZADO (Búsqueda + Juez)# ==========================================base_retriever = vector_db.as_retriever(search_kwargs={"k": 10})retriever_expandido = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)modelo_juez = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")compresor = CrossEncoderReranker(model=modelo_juez, top_n=3)retriever_filtrado = ContextualCompressionRetriever(    base_compressor=compresor,    base_retriever=retriever_expandido)def format_docs(docs):    return "\n\n".join(doc.page_content for doc in docs)# NUEVO: Función para formatear nuestro historial de chat a textodef formatear_historial(historial):    if not historial:        return "No hay historial previo."        texto_historial = ""    for mensaje in historial:        texto_historial += f"Usuario: {mensaje['usuario']}\nAsistente: {mensaje['asistente']}\n\n"    return texto_historial# ==========================================# FASE 3 Y MEMORIA: EL AGENTE CON HISTORIAL# ==========================================# 1. Actualizamos el Prompt del RAG para incluir el historialprompt_rag = PromptTemplate.from_template(    """Usa el siguiente contexto y el historial de la conversación para responder a la pregunta.     Si no lo sabes, di que no tienes información suficiente. No inventes.        Historial de conversación:    {chat_history}        Contexto recuperado:    {context}        Pregunta actual: {question}    Respuesta:""")# 2. Actualizamos el Prompt General para incluir el historialprompt_general = PromptTemplate.from_template(    """Eres un asistente amigable. Responde a la pregunta teniendo en cuenta el historial de la conversación.        Historial de conversación:    {chat_history}        Pregunta actual: {question}    Respuesta:""")# El clasificador se queda igual (solo necesita ver la pregunta actual para decidir)prompt_clasificador = PromptTemplate.from_template(    """Clasifica la siguiente entrada del usuario en UNA de estas dos categorías:    - 'repositorio': Si trata sobre código, variables, errores, archivos, el proyecto o hace referencia a algo técnico de lo que estéis hablando.    - 'general': Si es un saludo, charla trivial o pregunta genérica no técnica.        Responde SÓLO con la palabra de la categoría (repositorio o general).        Entrada: {question}    Categoría:""")cadena_clasificadora = prompt_clasificador | llm | StrOutputParser()# 3. Construimos las tuberías inyectando el historial dinámicamentecadena_rag = (    {"context": retriever_filtrado | format_docs,      "question": RunnablePassthrough(),     "chat_history": lambda x: formatear_historial(memoria_chat)} # Extrae la memoria global    | prompt_rag    | llm    | StrOutputParser())cadena_general = (    {"question": RunnablePassthrough(),     "chat_history": lambda x: formatear_historial(memoria_chat)} # Extrae la memoria global    | prompt_general    | llm    | StrOutputParser())# NUEVO: Lista global para almacenar la memoria de la sesiónmemoria_chat = []def agente_inteligente(pregunta: str):    print(f"\n[Usuario]: {pregunta}")    print("🤖 Analizando intención y leyendo historial...")        categoria = cadena_clasificadora.invoke({"question": pregunta}).strip().lower()        if "repositorio" in categoria:        print("🔍 [Ruta Técnica] Buscando en ChromaDB...")        respuesta = cadena_rag.invoke(pregunta)    else:        print("💬 [Ruta General] Respondiendo de memoria...")        respuesta = cadena_general.invoke(pregunta)            print(f"\n[Asistente]: {respuesta}\n")    print("-" * 50)        # NUEVO: Guardamos el turno en la memoria para la próxima pregunta    memoria_chat.append({"usuario": pregunta, "asistente": respuesta})        # Opcional: Limitar la memoria a los últimos 5 turnos para no saturar a Llama 3    if len(memoria_chat) > 5:        memoria_chat.pop(0)# ==========================================# BUCLE INTERACTIVO DE TERMINAL# ==========================================if __name__ == "__main__":    print("\n🚀 Sistema RAG 2.0 Agéntico Iniciado 🚀")    print("Escribe 'salir' para terminar la conversación.\n")    print("-" * 50)        # Ahora puedes chatear directamente desde la terminal en un bucle    while True:        entrada_usuario = input("\nTú: ")                if entrada_usuario.lower() in ['salir', 'exit', 'quit']:            print("¡Hasta luego! Cerrando el asistente.")            break                    if not entrada_usuario.strip():            continue                    agente_inteligente(entrada_usuario)

¿Qué ha cambiado en esta versión?variable memoria_chat: Hemos creado una lista simple de Python que guarda la pregunta y la respuesta tras cada interacción. Solo guardamos los últimos 5 turnos para que Llama 3 no se vuelva loco leyendo demasiada información antigua ni se ralentice tu portátil.Inyección en LCEL (chat_history): Usamos una pequeña función lambda dentro de las cadenas. Cada vez que le haces una pregunta, LangChain ejecuta formatear_historial(memoria_chat), coge los mensajes pasados, los convierte en un bloque de texto y se los pasa a Llama 3 como contexto adicional.Bucle while True: En lugar de lanzar preguntas de prueba estáticas, ahora tienes un chat interactivo real en la terminal.¡Ya tienes un sistema IA local con razonamiento, uso de herramientas (vectoriales), filtrado y memoria a corto plazo! Es una base excelente.